# Session 8 — Streaming Responses

Streaming is one of the key differences between a static notebook demo and a responsive application. This notebook shows the event pattern we will later reuse in Gradio and Streamlit.

## Learning Goals

- understand why streaming improves app experience
- use the Responses API with `stream=True`
- process text deltas from the event stream
- connect the same pattern to later UI examples

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')
print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

## Non-Streaming Baseline

Start with a normal Response API call so we can compare it to the streaming version.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response = client.responses.create(
    model='gpt-4o-mini',
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.'
)

display(Markdown(response.output_text))

## Streaming with the Responses API

Now we ask the same style of question, but receive output incrementally as events arrive.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response_stream = client.responses.create(
    model='gpt-4o-mini',
    instructions='You are a concise teaching assistant.',
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

streamed_text = ''
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for event in response_stream:
    if event.type == 'response.output_text.delta':
        streamed_text += event.delta
        handle.update(Markdown(streamed_text))

## Turn the Event Loop into a Generator

UI frameworks often want a generator or iterable. That makes streaming a natural fit.

In [ ]:
def stream_response(prompt: str, model: str = 'gpt-4o-mini'):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    stream = client.responses.create(
        model=model,
        instructions='You are a concise teaching assistant.',
        input=prompt,
        stream=True,
    )

    for event in stream:
        if event.type == 'response.output_text.delta':
            yield event.delta

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

collected = []
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for piece in stream_response('Give one short paragraph on why streamed output feels faster to users.'):
    collected.append(piece)
    handle.update(Markdown(''.join(collected)))

full_text = ''.join(collected)

display(Markdown('### Final combined text'))
display(Markdown(full_text))

## Why This Matters for the Rest of Session 8

- The Gradio demo can consume a generator like `stream_response(...)`.
- The Streamlit app can pass the same generator to `st.write_stream(...)`.
- Streaming is not a separate concept from app building; it is one of the main reasons the app feels interactive.